# Files into the conversation

Yesterday the agent could *list* your courses and files. It could not tell you
what was *inside* a file.

Today we give it one new power: read a file and pull its text into the chat. Once
the text is in the conversation, the model can summarize it, answer questions
about it, or compare two files.

We build three tools on top of the small `guc_cms` package:

- `list_courses` : what am I registered in
- `get_course_files` : what files are in a course
- `read_course_file` : read one file's content as text

## 1. Log in

The CMS uses your normal GUC username and password (Windows login).

This works for **GIU** as well: put `GUC_SITE=giu` in your `.env` and use your
GIU username and password. Nothing else in the notebook changes.

In [ ]:
import os, getpass
from dotenv import load_dotenv

load_dotenv()  # reads ANTHROPIC_API_KEY from .env (and GUC login if you put it there)

# Type your GUC login once. getpass hides the password so it is never saved
# in the notebook. Nothing here is written to disk.
if not os.environ.get("GUC_USERNAME"):
    os.environ["GUC_USERNAME"] = input("GUC username: ")
if not os.environ.get("GUC_PASSWORD"):
    os.environ["GUC_PASSWORD"] = getpass.getpass("GUC password: ")

from guc_cms import GucCms

cms = GucCms()
courses = cms.list_courses()
print(len(courses), "courses. First few:")
for c in courses[:5]:
    print("  ", c.code, "-", c.title)

## 2. Any file becomes text

The files are PDFs and Excel sheets. A model reads text, not PDFs. So we need a
translator: file bytes go in, markdown text comes out.

`markitdown` does exactly that for many file types with one call. Let us try it on
one file *before* we bring in the agent, so we can see the raw idea.

In [ ]:
from io import BytesIO
from markitdown import MarkItDown

converter = MarkItDown()

# grab one file from a course
content = cms.get_content(cms.find_course("Software Engineering"))
some_file = content.items[0]
print("file:", some_file.title, "|", some_file.filename)

raw = cms.fetch_bytes(some_file)                       # bytes in memory, nothing saved
ext = "." + some_file.filename.rsplit(".", 1)[-1].lower()
text = converter.convert_stream(BytesIO(raw), file_extension=ext).text_content

print("\nfirst 400 characters of the content:\n")
print(text[:400])

## 3. Wrap it as tools

Now the same idea, but as tools the agent can call on its own. Notice
`read_course_file` returns plain text. That text becomes part of the conversation,
so the model can reason about it.

We cut the text at 18000 characters. A tool result that is too long fills the
context window fast. Keep that number in mind, it comes back when we reach
backends.

In [ ]:
def _find_item(course, file_title):
    """Find one file in a course by a bit of its title. Returns a ContentItem or None."""
    c = cms.find_course(course)
    if not c:
        return None
    for item in cms.get_content(c).items:
        if file_title.lower() in item.title.lower():
            return item
    return None


from langchain.tools import tool


@tool
def list_courses() -> list[dict]:
    """List the student's registered courses (code and title)."""
    return [{"code": c.code, "title": c.title} for c in cms.list_courses() if c.active]


@tool
def get_course_files(course: str) -> list[dict]:
    """List the files in a course. `course` is a code or part of the name.
    Use this to find a file's exact title before reading it."""
    c = cms.find_course(course)
    if not c:
        return [{"error": f"no course matches {course!r}"}]
    return [{"title": i.title, "kind": i.kind, "week": i.week}
            for i in cms.get_content(c).items]


@tool
def read_course_file(course: str, file_title: str) -> str:
    """Read a course file's content as text, so you can answer questions about
    what is inside it. `file_title` can be part of the title."""
    item = _find_item(course, file_title)
    if not item:
        return f"No file matching {file_title!r} in {course!r}."
    raw = cms.fetch_bytes(item)
    ext = "." + item.filename.rsplit(".", 1)[-1].lower()
    text = converter.convert_stream(BytesIO(raw), file_extension=ext).text_content
    return text[:18000]

## 4. Give the tools to an agent

Same `create_agent` as yesterday. The only new part is the `tools` list.

In [ ]:
from langchain.agents import create_agent
from langchain_anthropic import ChatAnthropic
from langchain_core.messages import HumanMessage

model = ChatAnthropic(model="claude-haiku-4-5", temperature=0)

agent = create_agent(
    model=model,
    tools=[list_courses, get_course_files, read_course_file],
    system_prompt=(
        "You help a GUC student with their course materials. "
        "Use the tools to read files and answer about their content. "
        "Find the exact file with get_course_files before reading it."
    ),
)


def ask(question):
    result = agent.invoke({"messages": [HumanMessage(question)]})
    return result["messages"][-1].content


print(ask("In Software Engineering, what topics does the Final Exam Sample cover? List 3."))

## What just happened

The agent called `get_course_files` to find the file, then `read_course_file` to
pull its text into the conversation, then answered from that text.

The content is now *in context*: the model sees it the same way it sees your
question. That is powerful, but it has a cost. A 20 page PDF is thousands of
tokens. Do that for every file in every course and the context window overflows.

We will feel that limit soon. It is the reason the next step is a backend: a place
the agent can keep and open files without carrying all of them in the chat.

## Your turn

Try one of these:

1. Ask the agent to summarize a whole week of one course. It will read several files.
2. Add a tool `compare_files(course, title_a, title_b)` that reads two files and
   asks which is newer, or what changed between them.

We will add more here later. Keep this notebook, we build on it.